# GraphEnet HPE GNN Class Guide
This notebook walks through the main classes in `hpegnn.py`, focusing on how to use them, their inputs/outputs, and the logic flow. The emphasis is on `hpeGnn_splineConv` and `hpeGnn_splineConv_single_weight`.

## Big Picture
These classes implement Human Pose Estimation (13 joints → 26 coordinates) on graph data using PyTorch Lightning and PyTorch Geometric.
- **Inputs**: A graph with node features `x`, edges `edge_index`, edge attributes `edge_attr`, node positions `pos`, and a `batch` vector.
- **Outputs**: Predicted joint coordinates shaped `[num_graphs, 26]` (flattened `x, y` for 13 joints).
- **Training targets**: `data.y` (ground-truth joints), plus `data.th_pck` for PCK metric thresholding.
- **Metrics**: MSE loss, PCK, MPJPE.
- **Optimizers**: Adam with optional ReduceLROnPlateau scheduler.

## Shared Base: `hpegnn`
### Purpose
Base LightningModule providing training/validation/testing boilerplate, metric computation, video/csv export, and optimizer setup. It **does not** implement a forward pass—derived classes must provide it.

### Key init args
- `in_channels`: Node feature dimension (e.g., SCARF node features = 10).
- `hidden_channels`: Architecture-specific list (used by subclasses).
- `out_channels`: Number of joints (always 13 in this project).
- `learning_rate`, `batch_size`, `pck_multiplier`: Training hyperparameters.
- `visualise`, `image_size`, `save_video`, `write_csv`, `file_name_eval`: Visualization/export controls.

### Core methods
- `forward(...)`: Abstract placeholder.
- `basic_step(data)`:
  - Calls subclass `forward` → `(pred, node_repr)` (second value can be ignored).
        
  - Reshapes `data.y` to `[batch, 26]`.
        
  - Computes loss = `F.mse_loss(pred, y)`; metrics `pck_error`, `mpjpe_error`.
        
  - Returns `(loss, pck, mpjpe)`.
- `training_step` / `validation_step`: Log metrics via `self.log_dict`.
- `predict_step`: Runs `forward`, optionally writes CSV rows and/or video frames via `GraphVisualization`.
- `configure_optimizers`: Adam (+ optional ReduceLROnPlateau) with weight decay only on first conv by default.
- `custom_dot`: Helper to aggregate per-node joint predictions using dot products with node positions.

## GCN Variant: `hpegnn_gcnconv`
- Stack of `GCNConv` layers ending in `conv3` that outputs `2 * joints` channels.
- Uses `custom_dot` to pool node-wise logits into image-space joint coordinates.
- Dropout (0.1) between layers; `elu` activations.
- Optimizer: Adam with weight decay only on `conv1`.
- Use case: simpler baseline without spline edge attributes.

## Utility Variants
- `dummy_model`: For visualization only—renders ground truth, can optionally record video. No learning.
- `spline_gnn`: Three-layer SplineConv block with BatchNorm and optional dropout. Can serve as a feature extractor.
- `spline_module`: Single SplineConv + BatchNorm block, reused to build deeper stacks.

## Core Model: `hpeGnn_splineConv`
### What it is
- A spline-based GNN for pose regression.
- Builds a stack of `spline_module` blocks from `hidden_channels` (supports up to 10 layers).
- Final layer outputs `out_channels * 4` per node. For 13 joints → 52 channels, grouped as `(dx, dy, w_x, w_y)` per joint.

### Inputs
- `x_in`: Node features `[N_nodes, in_channels]` (first two dims are typically pixel positions from SCARF graphs).
- `edge_index`: Graph connectivity.
- `edge_attr`: Edge attributes (e.g., Cartesian offsets from `Cartesian()` transform).
- `batch`: Batch assignment vector.
- Hyperparameters: `hidden_channels`, `node_loss_weight`, `pck_multiplier`, `image_size`, `task`, etc.

### Forward flow
1. Clone `x_in` to `x`.
2. Pass through all child layers (the spline stack + `spline_last`).
3. `custom_softmax`: Applies edge-aware softmax on per-joint weights (`w_x`, `w_y`) over nodes within each graph in the batch.
4. `custom_sigmoid`: Squashes per-joint vector offsets `(dx, dy)` to `[-1, 1]`.
5. `custom_vect_dot`: For each joint and each graph:
        
   - Reshape outputs into vectors (`dx, dy`) and weights (`w_x, w_y`).
        
   - Convert normalized vectors to pixel scale via `image_size`.
        
   - Add to node positions to get candidate joint locations.
        
   - Weight and sum across nodes using `w_x, w_y` to produce final joint coordinates `[batch, 26]`.
6. Returns `(pred_joints, node_outputs)`.

### Loss options
- Base loss: MSE between predicted joints and `data.y`.
- Optional node-level loss (`node_loss_weight`): compares per-node joint proposals against the global target to encourage sharper node predictions. If `node_loss_weight` is a list, it weighs `[target_loss, node_loss]`; otherwise sums them.

### Outputs
- `pred`: `[batch, 26]` pixel coordinates.
- `node_outputs`: Raw per-node channels after spline layers (useful for visualization or auxiliary loss).

### Typical instantiation
```python
from graph_enet.hpe_gnn.model.hpegnn import hpeGnn_splineConv
model = hpeGnn_splineConv(
    in_channels=10,
    hidden_channels=[64, 64, 64],
    out_channels=13,
    learning_rate=1e-3,
    batch_size=8,
    node_loss_weight=[1.0, 0.2],
    pck_multiplier=0.6,
    image_size=[640, 480]
)
```

## Variant: `hpeGnn_splineConv_single_weight`
### What changes vs `hpeGnn_splineConv`
- Same spline stack construction, but weighting strategy differs: uses a **single weight per joint** (shared for x/y) instead of separate weights per axis.
- Output channels are still `out_channels * 4`, but grouping is `(dx, dy, w, w)` per joint (the two weight entries are identical indices).
- Pooling uses `custom_vect_dot_single_weight_vector`, which reuses the shared weight for both x/y contributions.
- Includes experimental helpers: `selective_pooling` (thresholding weights), `custom_softmax_last` (masked softmax). Currently the forward path uses standard `custom_softmax` + `custom_sigmoid`.

### Forward flow
1. Run spline stack on `x_in`.
2. Apply `custom_softmax` on weight channels (shared for x/y).
3. Apply `custom_sigmoid` on vector offsets `(dx, dy)` to keep them in `[-1, 1]`.
4. `custom_vect_dot_single_weight_vector`:
        
   - Extract `(dx, dy)` and weight `w` per joint per node.
        
   - Scale offsets by `image_size`, add to node positions to get proposals.
        
   - Weight-sum across nodes per joint to get final `[batch, 26]` coordinates.

### Loss options
- Same pattern: MSE on joints + optional node-level loss (`node_vect_loss`).

### When to pick this variant
- Prefer when you want a simpler weighting scheme (one weight per joint) or to match older checkpoints trained with this layout.

## Data and Shapes Recap
- `data.x`: `[N_nodes, in_channels]`, first two dims usually pixel coords.
- `data.edge_index`: `[2, N_edges]` (PyG COO).
- `data.edge_attr`: `[N_edges, 2]` (Cartesian offsets for SplineConv).
- `data.y`: `[batch, 1, 26]` or `[batch, 26]` → reshaped internally.
- `data.th_pck`: Threshold per sample for PCK calculation.
- Output `pred`: `[batch, 26]` → reshape to `[-1, 13, 2]` for visualization.

## Node Features by Dataset
- **SCARF spline graphs** (scarfDataset_splineConv → build_scarf_graph_splineConv): `x` has 10 dims `[x_mean, y_mean, RF_idx, v1_x, v1_y, v2_x, v2_y, lambda1, lambda2, eccentricity]` from PCA on events in each active RF. `pos` is the same `[x_mean, y_mean]`. `edge_attr` is `(Δx, Δy)` from `Cartesian(cat=False)` (normalized offsets). `y` is skeleton `[1, 2J]`; `th_pck` = distance between joints 2 and 6. Use SCARF params `rf_size=14, alpha=1.0, C=0.3, res=(640,480)`.
- **H36M GCN graphs** (customDatasets.eh36m_gcn): segments are 2-point lines with an energy value. Unique pixel points become nodes; `x` is a single scalar energy per node (sum if reused). `pos` stores pixel coords. `edge_index` links segment endpoints both directions; `edge_attr` is the segment energy duplicated. `th_pck` computed from torso diameter in the GT.
- **H36M spline graphs** (customDatasets.eh36m_spline_gamer / eh36m_spline_ledge): segments reshape to `(x0, y0, x1, y1, node_info...)`. `node_features` is 5 for gamer → one scalar per node; 6 for ledge → two scalars per node. First time a point is seen, its `node_info` is stored; reused points keep the first value. `edge_attr` is `(Δx, Δy)` from `Cartesian(norm=True, cat=False)` (normalized). Choose `in_channels` accordingly (e.g., 10 for SCARF, 1 or 2 for these H36M variants).

## Training Loop Integration
- Use PyTorch Lightning `Trainer` with standard `fit/validate/test/predict`.
- Metrics logged per step and per epoch: `loss`, `pck`, `mpjpe`.
- Scheduler: ReduceLROnPlateau monitors `loss/train`.
- Visualization: set `visualise` to `'pose'`, `'vectors-head'`, `'vectors-handR'`, or `None`.
- Video export: pass `save_video=path`.
- CSV export: provide `write_csv=dir` and `file_name_eval`.

## Practical Usage Snippet
```python
import torch
from torch_geometric.data import Data
from graph_enet.hpe_gnn.model.hpegnn import hpeGnn_splineConv_single_weight

# Dummy graph (replace with real SCARF graph)
x = torch.rand(120, 10)              # node features
edge_index = torch.randint(0, 120, (2, 400))
edge_attr = torch.rand(400, 2)       # Cartesian offsets
batch = torch.zeros(120, dtype=torch.long)
y = torch.rand(1, 26)                # target joints
th_pck = torch.tensor([25.0])        # example threshold
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, batch=batch, th_pck=th_pck)

model = hpeGnn_splineConv_single_weight(
    in_channels=10, hidden_channels=[64, 64, 64], out_channels=13, learning_rate=1e-3, batch_size=1
)
pred, _ = model.forward(data.x, data.edge_index, data.edge_attr, batch=data.batch)
print(pred.shape)  # torch.Size([1, 26])
```

## Tips and Gotchas
- Always ensure edge attributes exist: apply `torch_geometric.transforms.Cartesian(cat=False)` during graph construction.
- `image_size` must match the pixel space of your dataset (default 640×480).
- `out_channels` should stay at 13 to align with metrics and downstream tooling.
- If you enable `node_loss_weight`, tune its scale; start with `[1.0, 0.1]`.
- For visualization, keep batch size small to avoid heavy video writes.
- PCK uses `pck_multiplier` × `th_pck`; adjust `pck_multiplier` if thresholds feel too strict/lenient.

## Why the same model handles different node features
- `in_channels` is passed at model construction and must match the dataset’s `x` width. The spline models only assume that the **first two columns are pixel positions**; the remaining channels can vary by dataset (SCARF’s PCA stats vs. H36M energy scalars).
- The pooling math uses `x_in[:, 0:2]` as positions and the learned offsets/weights from the network head; everything else is just extra signal for the convolutions.
- Edge handling is consistent: all variants provide `edge_attr` as `(Δx, Δy)` (normalized when using `Cartesian(cat=False)`), so SplineConv always sees a 2D edge feature regardless of node schema.
- Dataset loaders pick different `in_channels`: SCARF graphs yield 10; H36M spline gamer yields 1; H36M spline ledge yields 2; the GCN variant can also accept these as long as you build the model with the matching `in_channels`.
- If `in_channels` mismatches the data, you’ll get shape errors at the first layer; otherwise the graph ops are agnostic to how many non-position features you provide.

### Are extra node features used?
- The spline models consume **all node channels** through the SplineConv stack; they influence the learned offsets/weights. Only the first two columns are explicitly reused at the end for coordinate reconstruction (`custom_vect_dot*` uses `x_in[:, 0:2]` as anchor positions).
- `edge_attr` `(Δx, Δy)` is always used inside SplineConv for spatial kernels.
- So: extra channels (PCA stats, energies, etc.) are not directly pooled into joints, but they shape the hidden activations that produce the per-joint vectors/weights. Mismatching or zeroing them changes predictions; they are meaningful context, not ignored.